# NucleiSky3D Benchmarking

## 🎯 Objective
This notebook benchmarks **NucleiSky3D** by repeatedly:
1. Taking **random 3D subvolumes** (patches) from a large reference volume.
2. Running NucleiSky3D to re-localize each patch inside the full volume.
3. Recording accuracy, robustness, and runtime across **patch sizes** and **matchers**.

It mirrors the logic of the 2D benchmarking notebook, but for the 3D pipeline.

---

## ✅ What you get
- **Comparable benchmark table** (CSV) across matchers and patch shapes
- Translation error (µm + voxel units), inlier fraction, mean registration error
- Optional intensity similarity check (SSIM on XY MIPs)
- Runtime profiling
- “Minimum nuclei required” analysis (success vs number of nuclei)

> **Tip:** For fair comparison, this notebook ensures *each matcher sees the same set of patch origins* per patch shape.


# 0. Install + import functions

In [ ]:
# @title  Initialize Environment & Install Dependencies

import os
import sys
from pathlib import Path

notebook_name = "NucleiSky3DBenchmarking"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------
if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'
        repo = 'nucleisky'
        # PEP 508 syntax: "package[extras] @ url"
        !pip install -q "nucleisky[all] @ git+https://{token}@github.com/{user}/{repo}.git"

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")
else:
    print("done")

# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
    print("    -> In Colab, go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")

# ---------------------------------------------------------
# 3. Package Compatibility Layer (nucleisky2d vs nucleiskyapp)
# ---------------------------------------------------------
for pkg_name in ["nucleisky2d", "nucleisky3d"]:
    try:
        pkg = importlib.import_module(pkg_name)
        # Create a dummy namespace for the umbrella package if needed
        if "nucleiskyapp" not in sys.modules:
            sys.modules["nucleiskyapp"] = type(sys)("nucleiskyapp")
        sys.modules[f"nucleiskyapp.{pkg_name}"] = pkg
    except ModuleNotFoundError:
        pkg = importlib.import_module(f"nucleiskyapp.{pkg_name}")
        sys.modules[pkg_name] = pkg


# ---------------------------------------------------------
# 4. Main Imports
# ---------------------------------------------------------
from IPython.display import display, clear_output
import ipywidgets as widgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import html
import json
import copy
import time
import zlib
import traceback

from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim

from nucleisky3d.demo_utils import generate_random_subvolume_3d
from nucleisky3d.io import (
    inspect_volume_header,
    load_volume,
    make_result_dir,
    require_voxel_size_um_zyx,
)

from nucleisky3d.features import extract_nuclear_features_3d
from nucleisky3d.preprocess import ij_percentile_normalize, scale_normalize_pair_for_segmentation
from nucleisky3d.segmentation import segment_nuclei_2p5d
from nucleisky3d.visualization import imshow_safe3d

from nucleisky3d.pipeline import NucleiSky3D, run_adaptive_nucleisky_3d
from nucleisky3d.config import DEFAULT_MATCHER_CONFIG, save_matcher_config_json

print("✅ Environment ready.")

In [ ]:
#@title Benchmark functions (3D)

# Columns written to CSV checkpoints and final table
BENCH_COLS_3D = [
    "matcher",
    "patch_z_px", "patch_y_px", "patch_x_px",
    "patch_z0", "patch_y0", "patch_x0",
    "n_nuclei",
    "success",
    "frac_inliers", "mean_error_um",
    "trans_error_um", "trans_error_px",
    "ssim_xy",
    "rot_error_deg",
    "best_scale",
    "time_matcher_s", "time_total_s",
]

# -----------------------------
# Utilities
# -----------------------------
def _sanitize_controller_kwargs_3d(d):
    """
    Allow ONLY keys that run_adaptive_nucleisky_3d actually expects as controller params.
    Everything else is dropped to avoid accidentally forwarding matcher kwargs.
    """
    if d is None:
        return {}
    if not isinstance(d, dict):
        raise ValueError("For matcher='adaptive', matcher_kwargs must be a dict of controller options.")

    allowed = {
        "matcher_order",
        "base_seed",
        "matcher_config",
        "stop_on_success",
        "store_full_out",
        "max_total_time_s",
        "verbose",
        "return_dists",
        "print_fn",
    }
    out = {k: v for k, v in d.items() if k in allowed}
    dropped = [k for k in d.keys() if k not in allowed]
    if dropped:
        print(f"ℹ️ adaptive: dropped unsupported controller kwargs: {dropped}")
    return out


def _clip_origin_zyx(z0, y0, x0, Z, Y, X, pz, py, px):
    z0 = int(max(0, min(int(z0), int(Z - pz))))
    y0 = int(max(0, min(int(y0), int(Y - py))))
    x0 = int(max(0, min(int(x0), int(X - px))))
    return z0, y0, x0


def _count_nuclei_in_patch_3d(zs_px, ys_px, xs_px, z0, y0, x0, pz, py, px):
    z1 = z0 + pz
    y1 = y0 + py
    x1 = x0 + px
    return int(np.sum(
        (zs_px >= z0) & (zs_px < z1) &
        (ys_px >= y0) & (ys_px < y1) &
        (xs_px >= x0) & (xs_px < x1)
    ))


def make_patch_candidates_anchored_on_nuclei_3d(
    Z, Y, X,
    patch_shape_px_zyx,
    zs_px, ys_px, xs_px,
    max_trials,
    seed,
    min_nuclei=1,
    oversample_factor=25,
):
    """
    Deterministically propose up to max_trials patch origins (z0,y0,x0) such that
    each patch contains at least min_nuclei nuclei (based on zs/ys/xs).
    """
    pz, py, px = [int(v) for v in patch_shape_px_zyx]
    max_trials = int(max_trials)
    min_nuclei = int(min_nuclei)

    if Z <= pz or Y <= py or X <= px:
        return np.zeros((0, 3), dtype=int)

    zs_px = np.asarray(zs_px, dtype=int)
    ys_px = np.asarray(ys_px, dtype=int)
    xs_px = np.asarray(xs_px, dtype=int)
    if zs_px.size == 0:
        return np.zeros((0, 3), dtype=int)

    rng = np.random.default_rng(int(seed))

    n_proposals = max(max_trials * int(oversample_factor), max_trials)
    idx = rng.integers(0, len(zs_px), size=n_proposals, endpoint=False)

    out = []
    seen = set()

    for i in idx:
        z = int(zs_px[i]); y = int(ys_px[i]); x = int(xs_px[i])

        # Place the chosen nucleus at a random location inside the patch
        oz = int(rng.integers(0, pz))
        oy = int(rng.integers(0, py))
        ox = int(rng.integers(0, px))

        z0, y0, x0 = _clip_origin_zyx(z - oz, y - oy, x - ox, Z, Y, X, pz, py, px)

        if (z0, y0, x0) in seen:
            continue

        if min_nuclei > 1:
            n_in = _count_nuclei_in_patch_3d(zs_px, ys_px, xs_px, z0, y0, x0, pz, py, px)
            if n_in < min_nuclei:
                continue

        seen.add((z0, y0, x0))
        out.append((z0, y0, x0))
        if len(out) >= max_trials:
            break

    return np.asarray(out, dtype=int)


import copy

def _deep_merge_dict(base: dict, override: dict | None) -> dict:
    """Recursively merge override into base without mutating either."""
    if override is None:
        return copy.deepcopy(base)
    out = copy.deepcopy(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge_dict(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def _stable_u32_from_str(s: str) -> int:
    """Stable across Python sessions (unlike built-in hash())."""
    return zlib.adler32(s.encode("utf-8")) & 0xFFFFFFFF


def make_benchmark_patch_plan_3d(
    vol_full_zyx,
    patch_shapes_px_zyx,
    patches_per_shape,
    seed=0,
    df_full=None,
    min_nuclei=1,
):
    """
    Returns dict: {shape_tuple: candidates_array(N,3) of (z0,y0,x0)}.

    If df_full is provided, candidates are anchored on nuclei and filtered
    to contain >= min_nuclei nuclei. Deterministic per (seed, shape).
    """
    vol_full_zyx = np.asarray(vol_full_zyx)
    if vol_full_zyx.ndim != 3:
        raise ValueError(f"Expected 3D volume (ZYX). Got shape={vol_full_zyx.shape}")

    Z, Y, X = vol_full_zyx.shape
    plan = {}

    zs_px = ys_px = xs_px = None
    if df_full is not None:
        needed = {"centroid_z_px", "centroid_y_px", "centroid_x_px"}
        if not needed.issubset(df_full.columns):
            raise ValueError(f"df_full must have {sorted(needed)} to enforce min_nuclei.")
        zs_px = df_full["centroid_z_px"].to_numpy(dtype=int, copy=False)
        ys_px = df_full["centroid_y_px"].to_numpy(dtype=int, copy=False)
        xs_px = df_full["centroid_x_px"].to_numpy(dtype=int, copy=False)

    for shape in patch_shapes_px_zyx:
        pz, py, px = [int(v) for v in shape]
        ss = np.random.SeedSequence([int(seed), int(pz), int(py), int(px), 13])
        shape_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        if df_full is not None:
            cand = make_patch_candidates_anchored_on_nuclei_3d(
                Z, Y, X,
                patch_shape_px_zyx=(pz, py, px),
                zs_px=zs_px, ys_px=ys_px, xs_px=xs_px,
                max_trials=int(patches_per_shape),
                seed=int(shape_seed),
                min_nuclei=int(min_nuclei),
            )
        else:
            # fallback: purely random origins (may include nucleus-free patches)
            rng = np.random.default_rng(int(shape_seed))
            n = int(patches_per_shape)
            if Z <= pz or Y <= py or X <= px:
                cand = np.zeros((0, 3), dtype=int)
            else:
                z0 = rng.integers(0, Z - pz + 1, size=n)
                y0 = rng.integers(0, Y - py + 1, size=n)
                x0 = rng.integers(0, X - px + 1, size=n)
                cand = np.column_stack([z0, y0, x0]).astype(int, copy=False)

        plan[(pz, py, px)] = cand

    return plan


def _benchmark_runtime_overrides_3d(matcher: str, base_overrides: dict | None, run_seed: int) -> dict:
    """
    Ensure NucleiSky3D receives deterministic _common.random_state=run_seed.
    Accepts:
      - hierarchical: {"_common": {...}, "pyramid": {...}}
      - flat: {"n_iters": 50000, ...} (assumed matcher-specific)
    """
    matcher_mode = str(matcher).strip().lower()
    # allow common typos
    if matcher_mode in ("hashing3d", "hash"):
        matcher_mode = "hashing"

    out = {"_common": {}, matcher_mode: {}}

    if base_overrides:
        if not isinstance(base_overrides, dict):
            raise ValueError("matcher_kwargs must be a dict (flat or hierarchical).")

        is_hier = ("_common" in base_overrides) or (matcher_mode in base_overrides)
        if is_hier:
            out = _deep_merge_dict(out, base_overrides)
        else:
            out[matcher_mode] = dict(base_overrides)

    out.setdefault("_common", {})
    out["_common"]["random_state"] = int(run_seed)
    return out


def _extract_quality_safe_3d(out: dict) -> tuple[float, float | None]:
    q = out.get("match_quality", {}) or {}
    try:
        frac = float(q.get("frac_inliers", 0.0) or 0.0)
    except Exception:
        frac = 0.0
    err = q.get("mean_error_um", None)
    try:
        if err is None:
            return frac, None
        err = float(err)
    except Exception:
        err = None
    return frac, err


def _rot_err_deg(R):
    if R is None:
        return None
    try:
        R = np.asarray(R, float).reshape(3, 3)
        tr = float(np.trace(R))
        c = (tr - 1.0) / 2.0
        c = float(np.clip(c, -1.0, 1.0))
        ang = float(np.arccos(c)) * (180.0 / np.pi)
        return ang
    except Exception:
        return None


def run_single_patch_match_benchmark_3d(
    vol_full_zyx,
    df_full,
    voxel_size_um_zyx,
    patch_shape_px_zyx,
    max_trials=30,
    random_state=0,
    compute_ssim=True,
    matcher="pyramid",

    matcher_config=None,
    matcher_kwargs=None,   # for adaptive: controller kwargs; for others: runtime overrides

    save_path=None,
    restart=False,
    candidates=None,
    candidates_seed=None,
    min_nuclei_skip=3,
    ij_percentile_normalize=None,
):
    """
    Benchmarks a single matcher on a set of candidate 3D patches.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize (used for SSIM calculation).")

    vol_full_zyx = np.asarray(vol_full_zyx)
    if vol_full_zyx.ndim != 3:
        raise ValueError(f"Benchmark expects a 3D volume (ZYX). Got shape={vol_full_zyx.shape}")

    Z, Y, X = vol_full_zyx.shape
    pz, py, px = [int(v) for v in patch_shape_px_zyx]

    voxel_size_um_zyx = tuple(float(v) for v in require_voxel_size_um_zyx(voxel_size_um_zyx))

    if Z <= pz or Y <= py or X <= px:
        print(f"Patch {patch_shape_px_zyx} is larger than volume {vol_full_zyx.shape}, skipping.")
        return []

    matcher_mode = str(matcher).strip().lower()
    if matcher_mode in ("hashing3d", "hash"):
        matcher_mode = "hashing"

    # ---- Candidates (fixed across matchers if provided) ----
    if candidates is None:
        seed = int(candidates_seed if candidates_seed is not None else (random_state if random_state is not None else 0))
        if df_full is not None and {"centroid_z_px","centroid_y_px","centroid_x_px"}.issubset(df_full.columns):
            zs = df_full["centroid_z_px"].to_numpy(dtype=int, copy=False)
            ys = df_full["centroid_y_px"].to_numpy(dtype=int, copy=False)
            xs = df_full["centroid_x_px"].to_numpy(dtype=int, copy=False)
            candidates = make_patch_candidates_anchored_on_nuclei_3d(
                Z, Y, X,
                patch_shape_px_zyx=(pz, py, px),
                zs_px=zs, ys_px=ys, xs_px=xs,
                max_trials=int(max_trials),
                seed=int(seed),
                min_nuclei=max(1, int(min_nuclei_skip)),
            )
        else:
            rng = np.random.default_rng(int(seed))
            z0 = rng.integers(0, Z - pz + 1, size=int(max_trials))
            y0 = rng.integers(0, Y - py + 1, size=int(max_trials))
            x0 = rng.integers(0, X - px + 1, size=int(max_trials))
            candidates = np.column_stack([z0, y0, x0]).astype(int, copy=False)
    else:
        candidates = np.asarray(candidates, dtype=int)

    # ---- Resume / checkpoint ----
    already_done_coords = set()
    rows = []

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        if save_path.exists() and (not restart):
            try:
                df_prev = pd.read_csv(save_path)
                for _, r in df_prev.iterrows():
                    already_done_coords.add((int(r["patch_z0"]), int(r["patch_y0"]), int(r["patch_x0"])))
                rows.extend(df_prev.to_dict("records"))
                print(f"🔁 Resuming from checkpoint: {save_path} ({len(rows)} rows loaded)")
            except Exception:
                print("⚠️ Could not read checkpoint, starting fresh.")

    def _append_and_checkpoint(metrics: dict, z0, y0, x0):
        rec = {k: metrics.get(k, None) for k in BENCH_COLS_3D}
        rec.update({
            "matcher": str(matcher_mode),
            "patch_z_px": int(pz), "patch_y_px": int(py), "patch_x_px": int(px),
            "patch_z0": int(z0), "patch_y0": int(y0), "patch_x0": int(x0),
        })
        rows.append(rec)

        if save_path is not None:
            try:
                pd.DataFrame(rows).to_csv(save_path, index=False)
            except Exception:
                pass

    # ---- Prepare full centroids ----
    if df_full is None:
        raise ValueError("df_full is required (must contain nuclei centroids).")

    needed_um = {"centroid_z_um", "centroid_y_um", "centroid_x_um"}
    if not needed_um.issubset(df_full.columns):
        if {"centroid_z_px","centroid_y_px","centroid_x_px"}.issubset(df_full.columns):
            df_full = df_full.copy()
            vz, vy, vx = voxel_size_um_zyx
            df_full["centroid_z_um"] = df_full["centroid_z_px"].astype(float) * vz
            df_full["centroid_y_um"] = df_full["centroid_y_px"].astype(float) * vy
            df_full["centroid_x_um"] = df_full["centroid_x_px"].astype(float) * vx
        else:
            raise ValueError("df_full must include either centroid_*_um or centroid_*_px columns.")

    centroids_full_um = df_full[["centroid_z_um","centroid_y_um","centroid_x_um"]].to_numpy(dtype=float, copy=False)

    # ---- Loop patches ----
    run_seed = int(random_state if random_state is not None else 0)

    for (z0, y0, x0) in tqdm(candidates, total=len(candidates), desc=f"{matcher_mode} {pz}x{py}x{px}"):
        z0 = int(z0); y0 = int(y0); x0 = int(x0)
        if (z0, y0, x0) in already_done_coords:
            continue

        ss = np.random.SeedSequence([int(run_seed), int(pz), int(py), int(px), int(z0), int(y0), int(x0), 999])
        patch_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        patch_start_t = time.perf_counter()

        z1 = z0 + pz; y1 = y0 + py; x1 = x0 + px
        if z1 > Z or y1 > Y or x1 > X:
            continue

        patch_vol = vol_full_zyx[z0:z1, y0:y1, x0:x1]

        # nuclei inside patch (prefer px coords)
        if {"centroid_z_px","centroid_y_px","centroid_x_px"}.issubset(df_full.columns):
            m_patch = (
                (df_full["centroid_z_px"] >= z0) & (df_full["centroid_z_px"] < z1) &
                (df_full["centroid_y_px"] >= y0) & (df_full["centroid_y_px"] < y1) &
                (df_full["centroid_x_px"] >= x0) & (df_full["centroid_x_px"] < x1)
            )
        else:
            vz, vy, vx = voxel_size_um_zyx
            z0u, y0u, x0u = z0*vz, y0*vy, x0*vx
            z1u, y1u, x1u = z1*vz, y1*vy, x1*vx
            m_patch = (
                (df_full["centroid_z_um"] >= z0u) & (df_full["centroid_z_um"] < z1u) &
                (df_full["centroid_y_um"] >= y0u) & (df_full["centroid_y_um"] < y1u) &
                (df_full["centroid_x_um"] >= x0u) & (df_full["centroid_x_um"] < x1u)
            )

        df_patch = df_full.loc[m_patch].copy()
        n_nuc = int(len(df_patch))

        if n_nuc < int(min_nuclei_skip):
            patch_total_t = time.perf_counter() - patch_start_t
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=0.0,
                mean_error_um=None,
                trans_error_um=None,
                trans_error_px=None,
                ssim_xy=None,
                rot_error_deg=None,
                best_scale=None,
                time_matcher_s=0.0,
                time_total_s=float(patch_total_t),
            ), z0, y0, x0)
            continue

        true_origin_um = np.array([z0*voxel_size_um_zyx[0], y0*voxel_size_um_zyx[1], x0*voxel_size_um_zyx[2]], dtype=float)
        centroids_crop_um = df_patch[["centroid_z_um","centroid_y_um","centroid_x_um"]].to_numpy(dtype=float, copy=False) - true_origin_um

        # ---- call matcher ----
        t0_match = time.perf_counter()

        inputs = dict(
            centroids_crop_um=centroids_crop_um,
            centroids_full_um=centroids_full_um,
            full_shape_px_zyx=tuple(vol_full_zyx.shape),
            crop_shape_px_zyx=(pz, py, px),
            pixel_size_full_um_zyx=voxel_size_um_zyx,
            pixel_size_crop_um_zyx=voxel_size_um_zyx,
            df_full=df_full,
            df_crop=df_patch,
        )

        out = None
        try:
            if matcher_mode == "adaptive":
                ctrl = _sanitize_controller_kwargs_3d(matcher_kwargs)
                ctrl = dict(ctrl)
                ctrl.setdefault("base_seed", int(patch_seed))
                if matcher_config is not None:
                    ctrl.setdefault("matcher_config", matcher_config)
                out, _records = run_adaptive_nucleisky_3d(**ctrl, **inputs)
            else:
                bench_overrides = _benchmark_runtime_overrides_3d(matcher_mode, matcher_kwargs, patch_seed)
                out = NucleiSky3D(
                    matcher=str(matcher_mode),
                    matcher_config=matcher_config,
                    matcher_kwargs=bench_overrides,
                    **inputs,
                )
        except Exception:
            traceback.print_exc()
            out = {}

        time_matcher_s = time.perf_counter() - t0_match

        if not isinstance(out, dict) or out.get("best_t", None) is None:
            patch_total_t = time.perf_counter() - patch_start_t
            frac_inliers, mean_error_um = _extract_quality_safe_3d(out if isinstance(out, dict) else {})
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=float(frac_inliers),
                mean_error_um=mean_error_um,
                trans_error_um=None,
                trans_error_px=None,
                ssim_xy=None,
                rot_error_deg=None,
                best_scale=None,
                time_matcher_s=float(time_matcher_s),
                time_total_s=float(patch_total_t),
            ), z0, y0, x0)
            continue

        best_t = np.asarray(out["best_t"], float).reshape(3,)
        q = out.get("match_quality", {}) or {}

        success_flag = bool(out.get("success", False))
        try:
            frac_inliers = float(q.get("frac_inliers", np.nan))
        except Exception:
            frac_inliers = np.nan
        mean_error_um = q.get("mean_error_um", None)

        delta_um = best_t - true_origin_um
        trans_error_um = float(np.linalg.norm(delta_um))
        delta_vox = delta_um / np.asarray(voxel_size_um_zyx, dtype=float)
        trans_error_px = float(np.linalg.norm(delta_vox))

        rot_err = _rot_err_deg(out.get("best_R", None))
        best_scale = out.get("best_scale", None)
        try:
            if best_scale is not None:
                best_scale = float(best_scale)
        except Exception:
            best_scale = None

        ssim_val = None
        if compute_ssim:
            z_pred0 = int(round(best_t[0] / voxel_size_um_zyx[0]))
            y_pred0 = int(round(best_t[1] / voxel_size_um_zyx[1]))
            x_pred0 = int(round(best_t[2] / voxel_size_um_zyx[2]))
            if (0 <= z_pred0 <= Z - pz) and (0 <= y_pred0 <= Y - py) and (0 <= x_pred0 <= X - px):
                pred_roi = vol_full_zyx[z_pred0:z_pred0+pz, y_pred0:y_pred0+py, x_pred0:x_pred0+px]
                patch_mip = np.max(patch_vol, axis=0)
                pred_mip = np.max(pred_roi, axis=0)
                patch_norm = ij_percentile_normalize(patch_mip)
                pred_norm  = ij_percentile_normalize(pred_mip)
                try:
                    ssim_val = float(ssim(patch_norm, pred_norm, data_range=1.0))
                except Exception:
                    ssim_val = None

        patch_total_t = time.perf_counter() - patch_start_t

        _append_and_checkpoint(dict(
            n_nuclei=n_nuc,
            success=bool(success_flag),
            frac_inliers=float(frac_inliers),
            mean_error_um=mean_error_um,
            trans_error_um=float(trans_error_um),
            trans_error_px=float(trans_error_px),
            ssim_xy=ssim_val,
            rot_error_deg=rot_err,
            best_scale=best_scale,
            time_matcher_s=float(time_matcher_s),
            time_total_s=float(patch_total_t),
        ), z0, y0, x0)

    return rows


def benchmark_patch_shapes_on_current_volume_multi(
    vol_full_zyx,
    df_full,
    voxel_size_um_zyx,
    patch_shapes_px_zyx=((16,128,128), (32,128,128), (32,192,192)),
    patches_per_shape=20,
    random_state=0,
    matchers=("pyramid", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map=None,  # can include "adaptive": {controller kwargs...}

    save_path_dir=None,
    restart=False,
    ij_percentile_normalize=None,

    min_nuclei=3,
):
    """
    Runs all matchers on the SAME candidate patches for each patch shape.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize.")

    if matcher_kwargs_map is None:
        matcher_kwargs_map = {}

    patch_plan = make_benchmark_patch_plan_3d(
        vol_full_zyx=vol_full_zyx,
        patch_shapes_px_zyx=patch_shapes_px_zyx,
        patches_per_shape=int(patches_per_shape),
        seed=int(random_state),
        df_full=df_full if int(min_nuclei) > 0 else None,
        min_nuclei=int(max(1, min_nuclei)),
    )

    all_rows = []

    for shape, candidates in patch_plan.items():
        pz, py, px = shape
        if len(candidates) == 0:
            continue

        for matcher in matchers:
            matcher_mode = str(matcher).strip().lower()
            if matcher_mode in ("hashing3d", "hash"):
                matcher_mode = "hashing"

            ckpt = None
            if save_path_dir is not None:
                save_dir = Path(save_path_dir)
                save_dir.mkdir(parents=True, exist_ok=True)
                ckpt = save_dir / f"{matcher_mode}_patch{pz}x{py}x{px}.csv"

            rows = run_single_patch_match_benchmark_3d(
                vol_full_zyx=vol_full_zyx,
                df_full=df_full,
                voxel_size_um_zyx=voxel_size_um_zyx,
                patch_shape_px_zyx=(pz, py, px),
                max_trials=int(patches_per_shape),
                random_state=int(random_state),
                compute_ssim=True,
                matcher=matcher_mode,

                matcher_config=matcher_config,
                matcher_kwargs=matcher_kwargs_map.get(matcher_mode, None),

                save_path=str(ckpt) if ckpt is not None else None,
                restart=bool(restart),

                candidates=candidates,
                candidates_seed=None,
                min_nuclei_skip=int(min_nuclei),

                ij_percentile_normalize=ij_percentile_normalize,
            )

            all_rows.extend(rows)

    df_results = pd.DataFrame(all_rows)
    return all_rows, df_results

# 1. Load your volume and extract features

In [ ]:
#@title Step 1: Choose 3D Volume + Voxel Size

# ----------------------------
# User inputs
# ----------------------------
# Path to a 3D TIFF (ZYX) or any format supported by nucleisky3d.io.load_volume
FULL_VOLUME_PATH = ""  # <-- set this (e.g. "/path/to/full_volume.tif")

# Voxel size in µm for the FULL volume (z, y, x).
# If your TIFF metadata is correct, you can leave this as None and it will be inferred when possible.
VOXEL_SIZE_UM_ZYX = None  # e.g. (0.3, 0.1, 0.1)

# Where to save results (folder will be created)
RESULT_DIR = "nucleisky3d_results"

# ----------------------------
# Optional UI helpers
# ----------------------------
path_w = widgets.Text(value=FULL_VOLUME_PATH, description="Full volume path:", layout=widgets.Layout(width="95%"))
vox_w  = widgets.Text(value="" if VOXEL_SIZE_UM_ZYX is None else ",".join(map(str, VOXEL_SIZE_UM_ZYX)),
                      description="Voxel size z,y,x (µm):",
                      layout=widgets.Layout(width="95%"))
out_w  = widgets.Text(value=str(RESULT_DIR), description="Result dir:", layout=widgets.Layout(width="95%"))

btn = widgets.Button(description="Apply", button_style="success")
out = widgets.Output()

def _parse_vox(s):
    s = (s or "").strip()
    if not s:
        return None
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if len(parts) != 3:
        raise ValueError("Voxel size must have 3 comma-separated values: z,y,x")
    return (float(parts[0]), float(parts[1]), float(parts[2]))

def _on_apply(_):
    global FULL_VOLUME_PATH, VOXEL_SIZE_UM_ZYX, RESULT_DIR
    with out:
        clear_output()
        FULL_VOLUME_PATH = path_w.value.strip()
        RESULT_DIR = out_w.value.strip() or "nucleisky3d_results"
        try:
            VOXEL_SIZE_UM_ZYX = _parse_vox(vox_w.value)
        except Exception as e:
            print("❌", e)
            return

        if not FULL_VOLUME_PATH:
            print("⚠️ Please set FULL_VOLUME_PATH.")
            return

        print("✅ Settings applied:")
        print("  FULL_VOLUME_PATH:", FULL_VOLUME_PATH)
        print("  VOXEL_SIZE_UM_ZYX:", VOXEL_SIZE_UM_ZYX)
        print("  RESULT_DIR:", RESULT_DIR)

        # Inspect header if possible
        try:
            hdr = inspect_volume_header(FULL_VOLUME_PATH)
            print("\n--- Header ---")
            print(hdr)
        except Exception as e:
            print("\n(Header inspection failed)", e)

btn.on_click(_on_apply)

display(widgets.VBox([path_w, vox_w, out_w, btn, out]))

In [ ]:
#@title Step 2: Segmentation (2.5D) + 3D Feature Extraction

# ----------------------------
# 1) Load full volume
# ----------------------------
if not FULL_VOLUME_PATH:
    raise ValueError("Please set FULL_VOLUME_PATH in Step 1.")

vol_full = load_volume(FULL_VOLUME_PATH)  # expected ZYX
vol_full = np.asarray(vol_full)
print("Full volume loaded:", vol_full.shape, vol_full.dtype)

# Infer voxel size if not provided
if VOXEL_SIZE_UM_ZYX is None:
    try:
        hdr = inspect_volume_header(FULL_VOLUME_PATH)
        VOXEL_SIZE_UM_ZYX = hdr.get("voxel_size_um_zyx", None)
    except Exception:
        VOXEL_SIZE_UM_ZYX = None

if VOXEL_SIZE_UM_ZYX is None:
    raise ValueError("Voxel size could not be inferred. Please set VOXEL_SIZE_UM_ZYX in Step 1 (z,y,x in µm).")

VOXEL_SIZE_UM_ZYX = tuple(float(v) for v in require_voxel_size_um_zyx(VOXEL_SIZE_UM_ZYX))
print("Voxel size (µm):", VOXEL_SIZE_UM_ZYX)

# Quick visualization: XY MIP
mip_xy = np.max(vol_full, axis=0)
plt.figure(figsize=(5,5), dpi=120)
plt.imshow(mip_xy, cmap="gray")
plt.title("XY MIP (full volume)")
plt.axis("off")
plt.show()

# ----------------------------
# 2) Segmentation (2.5D) or load precomputed labels
# ----------------------------
# If you already have a 3D label volume (ZYX) with integer labels, set this path:
LABELS_PATH = ""  # e.g. "/path/to/labels.tif" (optional)

# Otherwise, we run 2.5D segmentation (2D per-slice + stitching)
SEG_METHOD = "cellpose"  # {"cellpose", "instanseg", "threshold", ...}
SEG_SETTINGS = {
    # Cellpose / InstanSeg settings are passed through nucleisky2d.segment_nuclei_dispatch
    # Examples (cellpose):
    # "diameter": 40,
    # "channels": [0, 0],
    # "model_type": "cyto3",
}
MIN_IOU_STITCH = 0.3

if LABELS_PATH and Path(LABELS_PATH).exists():
    import tifffile
    labels_full = tifffile.imread(LABELS_PATH)
    labels_full = np.asarray(labels_full)
    print("Loaded labels:", labels_full.shape, labels_full.dtype)
else:
    print(f"Running 2.5D segmentation: method={SEG_METHOD}")
    labels_full = segment_nuclei_2p5d(
        volume_zyx=vol_full,
        method=SEG_METHOD,
        pixel_size_um_zyx=VOXEL_SIZE_UM_ZYX,
        settings=SEG_SETTINGS,
        min_iou=float(MIN_IOU_STITCH),
        show_progress=True,
    )
    print("Segmentation done:", labels_full.shape, labels_full.dtype)

# ----------------------------
# 3) Feature extraction (centroids + basic 3D features)
# ----------------------------
df_full = extract_nuclear_features_3d(labels_full, pixel_size_um=VOXEL_SIZE_UM_ZYX, k_neighbors=5)
print("Extracted nuclei:", len(df_full))
display(df_full.head())

# Optional: save metadata
RESULT_DIR = make_result_dir(RESULT_DIR)
meta = {
    "full_volume_path": FULL_VOLUME_PATH,
    "voxel_size_um_zyx": VOXEL_SIZE_UM_ZYX,
    "n_nuclei": int(len(df_full)),
}
with open(Path(RESULT_DIR) / "inputs_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("✅ Saved inputs metadata to:", Path(RESULT_DIR) / "inputs_metadata.json")

# 2. Run the 3D benchmark

In [ ]:
#@title Step 3: Run the Benchmark (3D)

# ---------------------------------------------------------
# DYNAMIC PATH SETUP
# ---------------------------------------------------------
base_output_dir = Path(RESULT_DIR) if ("RESULT_DIR" in globals() and RESULT_DIR) else Path("nucleisky3d_results")
output_dir = base_output_dir / "benchmarking"
output_dir.mkdir(parents=True, exist_ok=True)

base_name = Path(FULL_VOLUME_PATH).stem if FULL_VOLUME_PATH else "current_volume"

print(f"📂 Saving benchmark results to: {output_dir}")
print(f"🏷️  Base filename: {base_name}")

# ---------------------------------------------------------
# PREPARE DATA
# ---------------------------------------------------------
if "vol_full" not in globals() or vol_full is None:
    raise RuntimeError("Volume not found. Run Step 2 first.")
if "df_full" not in globals() or df_full is None:
    raise RuntimeError("df_full not found. Run Step 2 first.")
if VOXEL_SIZE_UM_ZYX is None:
    raise RuntimeError("VOXEL_SIZE_UM_ZYX not found. Run Step 1/2 first.")

vol_for_bench = vol_full
vox_for_bench = VOXEL_SIZE_UM_ZYX

print("\nBenchmark settings:")
print(f"  Volume shape: {np.asarray(vol_for_bench).shape}")
print(f"  Voxel size:   {vox_for_bench} µm (z,y,x)")
print(f"  N nuclei:     {len(df_full)}")

# ---------------------------------------------------------
# RUN BENCHMARK
# ---------------------------------------------------------
# Patch shapes are (Z, Y, X) in pixels.
# Use smaller Z for anisotropic stacks; keep Y/X similar to your typical crop sizes.
PATCH_SHAPES_PX_ZYX = [
    (16, 128, 128),
    (32, 128, 128),
    (32, 192, 192),
    (48, 192, 192),
]
PATCHES_PER_SHAPE = 30
RANDOM_STATE = 0

MATCHERS = ("pyramid", "hashing", "adaptive")

# Optional: tweak matcher_config here (otherwise defaults are used)
matcher_config = None  # e.g. copy.deepcopy(DEFAULT_MATCHER_CONFIG)

matcher_kwargs_map = {
    # Adaptive controller options (NOT matcher runtime kwargs)
    "adaptive": {
        "matcher_order": ["pyramid", "hashing"],
        "stop_on_success": True,
        "max_total_time_s": None,
        "verbose": False,
    },

    # Example runtime override for pyramid (if you want):
    # "pyramid": {"pyramid": {"n_iters": 50000}},
    # Example runtime override for hashing:
    # "hashing": {"hashing": {"max_candidates": 2000}},
}

MIN_NUCLEI_PER_PATCH = 3

all_results, df_results = benchmark_patch_shapes_on_current_volume_multi(
    vol_full_zyx=vol_for_bench,
    df_full=df_full,
    voxel_size_um_zyx=vox_for_bench,

    patch_shapes_px_zyx=PATCH_SHAPES_PX_ZYX,
    patches_per_shape=int(PATCHES_PER_SHAPE),
    random_state=int(RANDOM_STATE),

    matchers=MATCHERS,
    matcher_config=matcher_config,
    matcher_kwargs_map=matcher_kwargs_map,

    min_nuclei=int(MIN_NUCLEI_PER_PATCH),

    save_path_dir=str(output_dir / f"{base_name}_checkpoints"),
    restart=False,

    ij_percentile_normalize=ij_percentile_normalize,
)

# ---------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------
csv_path = output_dir / f"{base_name}_benchmark3d_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"\n✅ Saved final benchmark CSV to:\n   {csv_path}")

display(df_results.head())

# 3. Plot results

In [ ]:
#@title Plots

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _sanitize_df_3d(df_results):
    if not isinstance(df_results, pd.DataFrame):
        df_results = pd.DataFrame(df_results)
    df = df_results.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in t if str(x) != ""]).strip() for t in df.columns]
    df.columns = [str(c).strip() for c in df.columns]

    required = [
        "matcher",
        "patch_z_px","patch_y_px","patch_x_px",
        "patch_z0","patch_y0","patch_x0",
        "n_nuclei","success",
        "frac_inliers","trans_error_px",
        "time_matcher_s","time_total_s",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}. Available: {list(df.columns)}")

    df["matcher"] = df["matcher"].astype(str)
    for c in ["patch_z_px","patch_y_px","patch_x_px","patch_z0","patch_y0","patch_x0"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    df["n_nuclei"] = pd.to_numeric(df["n_nuclei"], errors="coerce").fillna(0).astype(int)
    df["success"] = df["success"].astype(bool)
    df["frac_inliers"] = pd.to_numeric(df["frac_inliers"], errors="coerce")
    df["trans_error_px"] = pd.to_numeric(df["trans_error_px"], errors="coerce")
    df["time_matcher_s"] = pd.to_numeric(df["time_matcher_s"], errors="coerce")
    df["time_total_s"] = pd.to_numeric(df["time_total_s"], errors="coerce")
    if "ssim_xy" in df.columns:
        df["ssim_xy"] = pd.to_numeric(df["ssim_xy"], errors="coerce")

    df["patch_label"] = (
        df["patch_z_px"].astype(str) + "x" +
        df["patch_y_px"].astype(str) + "x" +
        df["patch_x_px"].astype(str)
    )
    return df

df = _sanitize_df_3d(df_results)

# 1) Success rate by patch shape
succ = df.groupby(["matcher","patch_label"], observed=True)["success"].mean().reset_index()
pivot = succ.pivot(index="patch_label", columns="matcher", values="success").sort_index()

plt.figure(figsize=(10, 4), dpi=120)
for m in pivot.columns:
    plt.plot(pivot.index, pivot[m].values, marker="o", label=m)
plt.ylim(-0.02, 1.02)
plt.ylabel("Success rate")
plt.xlabel("Patch shape (ZxYxX px)")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("NucleiSky3D Success rate vs Patch shape")
plt.tight_layout()
plt.show()

# 2) Translation error (voxels) on successful matches
df_ok = df[df["success"] & df["trans_error_px"].notna()].copy()
if len(df_ok):
    err = df_ok.groupby(["matcher","patch_label"], observed=True)["trans_error_px"].median().reset_index()
    pivot2 = err.pivot(index="patch_label", columns="matcher", values="trans_error_px").sort_index()

    plt.figure(figsize=(10, 4), dpi=120)
    for m in pivot2.columns:
        plt.plot(pivot2.index, pivot2[m].values, marker="o", label=m)
    plt.ylabel("Median translation error (voxels)")
    plt.xlabel("Patch shape (ZxYxX px)")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.title("Translation error on successful matches")
    plt.tight_layout()
    plt.show()

# 3) Runtime
rt = df.groupby(["matcher","patch_label"], observed=True)["time_matcher_s"].median().reset_index()
pivot3 = rt.pivot(index="patch_label", columns="matcher", values="time_matcher_s").sort_index()

plt.figure(figsize=(10, 4), dpi=120)
for m in pivot3.columns:
    plt.plot(pivot3.index, pivot3[m].values, marker="o", label=m)
plt.ylabel("Median matcher runtime (s)")
plt.xlabel("Patch shape (ZxYxX px)")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("Runtime vs Patch shape")
plt.tight_layout()
plt.show()

# 4) SSIM (optional)
if "ssim_xy" in df.columns:
    df_ok2 = df[df["success"] & df["ssim_xy"].notna()].copy()
    if len(df_ok2):
        s = df_ok2.groupby(["matcher","patch_label"], observed=True)["ssim_xy"].median().reset_index()
        pivot4 = s.pivot(index="patch_label", columns="matcher", values="ssim_xy").sort_index()

        plt.figure(figsize=(10, 4), dpi=120)
        for m in pivot4.columns:
            plt.plot(pivot4.index, pivot4[m].values, marker="o", label=m)
        plt.ylim(-0.02, 1.02)
        plt.ylabel("Median SSIM (XY MIP)")
        plt.xlabel("Patch shape (ZxYxX px)")
        plt.xticks(rotation=45, ha="right")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.title("Intensity similarity (SSIM on XY MIPs)")
        plt.tight_layout()
        plt.show()

In [ ]:
#@title Minimum nuclei requirement (success vs N nuclei)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

df = _sanitize_df_3d(df_results)

# Define "correct" match
SUCCESS_DIST_VOX = 10.0  # translation error tolerance in voxel units (Euclidean in z,y,x voxels)
df["is_correct"] = df["success"] & df["trans_error_px"].notna() & (df["trans_error_px"] < SUCCESS_DIST_VOX)

def sigmoid(x, k, x0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))

plt.figure(figsize=(10, 5), dpi=120)

summary = []

matchers = sorted(df["matcher"].unique())
for i, matcher in enumerate(matchers):
    grp = df[df["matcher"] == matcher].copy()
    if grp.empty:
        continue

    x = grp["n_nuclei"].values.astype(float)
    y = grp["is_correct"].values.astype(float)

    # Bin for visualization
    bins = np.arange(0, max(10, int(np.nanmax(x)) + 5), 5)
    grp["bin"] = pd.cut(grp["n_nuclei"], bins=bins)
    binned = grp.groupby("bin", observed=True).agg({"is_correct":"mean", "n_nuclei":"mean"}).dropna()

    plt.scatter(binned["n_nuclei"], binned["is_correct"], s=25, alpha=0.5, label=f"{matcher} (binned)")

    # Fit curve
    try:
        popt, _ = curve_fit(sigmoid, x, y, p0=[0.2, np.median(x)], maxfev=20000)
        k, x0 = popt
        xs = np.linspace(0, max(5, np.nanmax(x) + 5), 200)
        ys = sigmoid(xs, k, x0)
        plt.plot(xs, ys, linewidth=2)

        # N at 50% and 90%
        n50 = float(x0)
        n90 = float(x0 + np.log(9)/k) if k != 0 else np.inf
        summary.append({"matcher": matcher, "k": float(k), "x0(n50)": n50, "n90": n90})
    except Exception:
        summary.append({"matcher": matcher, "k": np.nan, "x0(n50)": np.nan, "n90": np.nan})

plt.axhline(0.9, linestyle="--", linewidth=1)
plt.ylim(-0.02, 1.02)
plt.xlabel("Number of nuclei in patch")
plt.ylabel(f"P(correct match)   [correct = success & error<{SUCCESS_DIST_VOX:.1f} vox]")
plt.title("Minimum nuclei requirement (logistic fit)")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary).sort_values("matcher")
display(summary_df)